# **Fase 1:** Informe Casos Semanales de Dengue en Cali
## Universidad Autónoma de Occidente
### Pronóstico de Series Temporales, Profesor: Sergio Alejandro Cantillo Luna.
### Marko David García, Alejandro Meneses Portilla.

**Objetivo del notebook:** Preparar una serie temporal semanal de casos de dengue, revisar su calidad, construir modelos baseline, evaluar sus errores y diagnosticar si todavía quedan patrones temporales en los residuos.

### Resumen General

Este notebook aborda el problema como un ejercicio completo de analítica de series temporales epidemiológicas. A partir de registros individuales de dengue, se construye una serie semanal en formato Nixtla, se valida su continuidad temporal, se diagnostican outliers y posibles cambios de régimen, y se comparan modelos baseline antes de avanzar hacia modelos más sofisticados.

Con los datos disponibles, la serie contiene **61.933 registros individuales** agregados en **744 semanas**, desde la semana que inicia el **28 de diciembre de 2009** hasta la semana que inicia el **25 de marzo de 2024**. La media semanal es cercana a **83 casos**, pero la mediana es **51**, lo que evidencia una distribución asimétrica: muchas semanas de baja o moderada transmisión conviven con brotes epidémicos de gran magnitud. Los picos principales se concentran en **2010**, **2020** y **2023**, años que deben interpretarse como episodios epidemiológicos relevantes y no como simples anomalías estadísticas.

La lectura central es que el dengue presenta **persistencia temporal fuerte**, alta variabilidad y episodios de cambio de nivel. Por tanto, los baselines son útiles como punto de comparación, pero no deben considerarse el modelo final. El valor del ejercicio está en construir una línea base defendible, identificar qué estructura queda sin explicar y justificar el paso posterior hacia modelos con estacionalidad, covariables climáticas y/o dinámica no lineal.


---
## ⚙️ PARTE 1: Configuración del Entorno

En esta sección se instalan y cargan las librerías necesarias para el análisis. Se dejan definidas las rutas, constantes y parámetros principales del experimento.

La frecuencia `W-MON` indica que cada observación representa la semana que inicia un lunes, lo cual es consistente con la construcción ISO de semanas epidemiológicas. La fecha de corte `2020-01-01` separa un período de entrenamiento largo, con múltiples ciclos epidémicos, de un período de prueba desafiante que incluye años recientes con cambios importantes en la dinámica de transmisión.

### **Nota de reproducibilidad:** las rutas del notebook están escritas en formato local (`/...`). Si se ejecuta en COLAB, basta con ajustar `RUTA_DATOS_CRUDOS` y `RUTA_SALIDA_NIXTLA` a los archivos del proyecto.


In [1]:
!pip install statsforecast utilsforecast statsmodels -q

In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from scipy import stats
from statsmodels.tsa.stattools import acf, adfuller, kpss
from statsmodels.tsa.seasonal import STL
from statsmodels.stats.diagnostic import acorr_ljungbox

from statsforecast import StatsForecast
from statsforecast.models import Naive, SeasonalNaive, WindowAverage, RandomWalkWithDrift

from utilsforecast.losses import mae, rmse, smape
from utilsforecast.evaluation import evaluate

import warnings
warnings.filterwarnings('ignore')

# =========================
# Parámetros principales
# =========================
RUTA_DATOS_CRUDOS = "datos_dengue_202604282143.csv"
RUTA_SALIDA_NIXTLA = "dengue_anio_semana_nixtla.csv"

UNIQUE_ID = "dengue_cali"
COLUMNA_FECHA = "fec_not"
FRECUENCIA = "W-MON"
FECHA_CORTE = "2020-01-01"
ESTACIONALIDAD_CUATRIENAL = 52 * 4

MODELOS_BASELINE = ["Naive", "SeasonalNaive", "WindowAverage", "RWD"]

print("✅ Todas las librerías cargadas correctamente")


✅ Todas las librerías cargadas correctamente


/home/alejo/.virtualenvs/datascience-venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## 🧰 PARTE 1.1: Funciones Reutilizables

Para mantener el notebook ordenado, las operaciones repetidas se encapsulan en funciones.

In [3]:
def leer_datos_dengue(ruta_archivo):
    """Carga el archivo original de dengue."""
    datos = pd.read_csv(ruta_archivo)
    print("Primeras filas del dataset original:")
    display(datos.head())
    print("\nColumnas disponibles:")
    print(datos.columns.tolist())
    return datos


def construir_serie_semanal_nixtla(datos, columna_fecha="fec_not", unique_id="dengue_cali"):
    """
    Convierte el dataset diario/individual en una serie semanal con formato Nixtla:
    unique_id, ds, y.
    """
    datos = datos.copy()
    datos[columna_fecha] = pd.to_datetime(datos[columna_fecha], errors="coerce")
    datos = datos.dropna(subset=[columna_fecha])

    calendario_iso = datos[columna_fecha].dt.isocalendar()
    datos["anio"] = calendario_iso.year.astype(int)
    datos["semana"] = calendario_iso.week.astype(int)

    # Lunes de cada semana ISO. Este formato evita ambigüedades entre año calendario y año ISO.
    datos["ds"] = pd.to_datetime(
        datos["anio"].astype(str) + "-W" + datos["semana"].astype(str).str.zfill(2) + "-1",
        format="%G-W%V-%u"
    )

    serie = (
        datos.groupby("ds")
        .size()
        .reset_index(name="y")
        .sort_values("ds")
        .reset_index(drop=True)
    )

    serie.insert(0, "unique_id", unique_id)
    serie = serie[["unique_id", "ds", "y"]]
    return serie


def guardar_serie_nixtla(serie, ruta_salida):
    """Guarda la serie semanal en formato Nixtla."""
    serie.to_csv(ruta_salida, index=False)
    print("Archivo generado correctamente:")
    print(ruta_salida)


def graficar_serie(serie, titulo="Número de Casos de Dengue por Semana"):
    """Grafica la serie temporal completa."""
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=serie["ds"],
            y=serie["y"],
            mode="lines+markers",
            name="Casos de Dengue",
            line=dict(color="#2196F3", width=1.8),
            marker=dict(size=4)
        )
    )
    fig.update_layout(
        title=titulo,
        xaxis_title="Semana Epidemiológica",
        yaxis_title="Número de Casos",
        height=450,
        template="plotly_white"
    )
    fig.show()



def graficar_heatmap_anio_semana(serie):
    """Mapa de calor año-semana para visualizar intensidad de brotes."""
    serie_plot = serie.copy()
    calendario = serie_plot.ds.dt.isocalendar()
    serie_plot["anio"] = calendario.year.astype(int)
    serie_plot["semana"] = calendario.week.astype(int)

    matriz = serie_plot.pivot_table(
        index="anio",
        columns="semana",
        values="y",
        aggfunc="sum",
        fill_value=0
    ).sort_index()

    fig = go.Figure(data=go.Heatmap(
        z=matriz.values,
        x=matriz.columns,
        y=matriz.index,
        colorscale="YlOrRd",
        colorbar=dict(title="Casos"),
        hovertemplate="Año=%{y}<br>Semana=%{x}<br>Casos=%{z}<extra></extra>"
    ))
    fig.update_layout(
        title="Mapa de Calor Año-Semana — Casos de Dengue",
        xaxis_title="Semana epidemiológica",
        yaxis_title="Año",
        height=430,
        template="plotly_white"
    )
    fig.show()


def graficar_descomposicion_stl(serie, periodo=ESTACIONALIDAD_CUATRIENAL):
    """Descomposición STL en tendencia, estacionalidad y residuo."""
    serie_stl = serie.sort_values("ds").set_index("ds")["y"].astype(float)
    resultado = STL(serie_stl, period=periodo, robust=True).fit()

    fig = make_subplots(
        rows=4,
        cols=1,
        shared_xaxes=True,
        subplot_titles=["Serie original", "Tendencia", "Estacionalidad", "Residuo"],
        vertical_spacing=0.06
    )
    componentes = [serie_stl, resultado.trend, resultado.seasonal, resultado.resid]
    colores = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0"]

    for fila, valores, color in zip(range(1, 5), componentes, colores):
        fig.add_trace(
            go.Scatter(x=valores.index, y=valores.values, mode="lines", line=dict(color=color, width=1.5)),
            row=fila,
            col=1
        )

    fig.update_layout(
        title=f"Descomposición STL — periodo estacional = {periodo} semanas",
        height=720,
        template="plotly_white",
        showlegend=False
    )
    fig.update_yaxes(title_text="Casos", row=1, col=1)
    fig.update_yaxes(title_text="Tendencia", row=2, col=1)
    fig.update_yaxes(title_text="Estacional", row=3, col=1)
    fig.update_yaxes(title_text="Residuo", row=4, col=1)
    fig.show()
    return resultado


def completar_calendario_semanal(serie, frecuencia="W-MON", unique_id="dengue_cali"):
    """Crea el calendario semanal completo y deja NaN donde falten semanas."""
    rango_completo = pd.date_range(serie.ds.min(), serie.ds.max(), freq=frecuencia)
    calendario = pd.DataFrame({"ds": rango_completo, "unique_id": unique_id})
    serie_completa = calendario.merge(serie, on=["unique_id", "ds"], how="left")
    return serie_completa[["unique_id", "ds", "y"]]


def diagnosticar_faltantes(serie, frecuencia="W-MON"):
    """Revisa continuidad temporal y valores faltantes en y."""
    fechas_esperadas = pd.date_range(serie.ds.min(), serie.ds.max(), freq=frecuencia)
    fechas_faltantes = fechas_esperadas.difference(serie.ds)

    print("=" * 55)
    print("  DIAGNÓSTICO DE VALORES FALTANTES")
    print("=" * 55)
    print(f"  Observaciones esperadas: {len(fechas_esperadas)}")
    print(f"  Observaciones presentes: {len(serie)}")
    print(f"  Fechas faltantes:        {len(fechas_faltantes)}")
    print(f"  NaN en columna y:        {serie.y.isna().sum()}")

    if len(fechas_faltantes) == 0 and serie.y.isna().sum() == 0:
        print("\n  ✅ Serie completa — sin valores faltantes")
    else:
        print(f"\n  ⚠️  Fechas faltantes: {fechas_faltantes.tolist()}")

    return fechas_faltantes


def separar_train_test(serie, fecha_corte):
    """Hace split temporal. NO se hace split aleatorio."""
    fecha_corte = pd.to_datetime(fecha_corte)
    train = serie[serie.ds < fecha_corte].copy()
    test = serie[serie.ds >= fecha_corte].copy()
    return train, test


def graficar_split(train, test, fecha_corte):
    """Grafica el conjunto de entrenamiento y prueba."""
    proporcion_train = len(train) / (len(train) + len(test)) * 100
    fecha_corte = pd.to_datetime(fecha_corte)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=train.ds, y=train.y, mode="lines+markers",
        name="Train", line=dict(color="#2196F3", width=2), marker=dict(size=3)
    ))
    fig.add_trace(go.Scatter(
        x=test.ds, y=test.y, mode="lines+markers",
        name="Test", line=dict(color="#FF5722", width=2), marker=dict(size=3)
    ))
    fig.add_shape(
        type="line", xref="x", yref="paper",
        x0=fecha_corte, x1=fecha_corte, y0=0, y1=1,
        line=dict(dash="dash", color="gray", width=1.5)
    )
    fig.add_annotation(
        x=fecha_corte, y=1.02, yref="paper",
        text="Corte", showarrow=False,
        font=dict(size=10, color="gray"), xanchor="left"
    )
    fig.update_layout(
        title=f"Train-Test Split Temporal — {proporcion_train:.0f}% / {100 - proporcion_train:.0f}%",
        xaxis_title="Fecha", yaxis_title="Casos",
        height=380, template="plotly_white"
    )
    fig.show()


def calcular_medias_estacionales(train):
    """Calcula la media por semana del año usando únicamente el train."""
    train_aux = train.copy()
    train_aux["semana_anio"] = train_aux.ds.dt.isocalendar().week.astype(int)
    medias = train_aux.groupby("semana_anio")["y"].mean().to_dict()
    media_global = train_aux["y"].mean()
    return medias, media_global


def imputar_por_media_estacional(datos, medias_estacionales, media_global):
    """Imputa NaN usando la media estacional calculada previamente."""
    datos = datos.copy()
    datos["semana_anio"] = datos.ds.dt.isocalendar().week.astype(int)

    def imputar_fila(fila):
        if pd.isna(fila["y"]):
            return medias_estacionales.get(fila["semana_anio"], media_global)
        return fila["y"]

    datos["y"] = datos.apply(imputar_fila, axis=1)
    datos = datos.drop(columns=["semana_anio"])
    return datos


In [4]:
def detectar_outliers_iqr(serie):
    """Detecta outliers usando el método IQR global."""
    q1, q3 = serie.y.quantile(0.25), serie.y.quantile(0.75)
    iqr = q3 - q1
    limite_inf = q1 - 1.5 * iqr
    limite_sup = q3 + 1.5 * iqr
    outliers = serie[(serie.y < limite_inf) | (serie.y > limite_sup)].copy()
    return outliers, q1, q3, iqr, limite_inf, limite_sup


def detectar_outliers_zscore(serie, umbral=3):
    """Detecta outliers usando Z-score."""
    z_scores = np.abs(stats.zscore(serie.y))
    return serie[z_scores > umbral].copy()


def graficar_outliers_iqr(serie, outliers, limite_inf, limite_sup):
    """Grafica la serie con límites IQR."""
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=serie.ds, y=serie.y, mode="lines",
        name="Serie", line=dict(color="#2196F3", width=1.5)
    ))
    fig.add_hline(y=limite_sup, line_dash="dash", line_color="orange", annotation_text="Límite IQR superior")
    fig.add_hline(y=limite_inf, line_dash="dash", line_color="orange", annotation_text="Límite IQR inferior")

    if len(outliers):
        fig.add_trace(go.Scatter(
            x=outliers.ds, y=outliers.y,
            mode="markers", name="Outlier (IQR)",
            marker=dict(color="red", size=5, symbol="circle-open", line_width=1)
        ))

    fig.update_layout(
        title="Detección de Outliers — Método IQR",
        height=380, template="plotly_white",
        xaxis_title="Fecha", yaxis_title="Casos"
    )
    fig.show()


def detectar_outliers_iqr_por_mes(serie):
    """Detecta outliers aplicando IQR dentro de cada mes."""
    df_check = serie.copy()
    df_check["mes"] = df_check.ds.dt.month
    df_check["outlier_mensual"] = False

    for mes in range(1, 13):
        mascara = df_check.mes == mes
        valores_mes = df_check.loc[mascara, "y"]
        q1, q3 = valores_mes.quantile(0.25), valores_mes.quantile(0.75)
        iqr_mes = q3 - q1
        es_outlier = (valores_mes < q1 - 1.5 * iqr_mes) | (valores_mes > q3 + 1.5 * iqr_mes)
        df_check.loc[mascara & es_outlier, "outlier_mensual"] = True

    return df_check


def graficar_cambios_regimen(serie, ventana=2):
    """Grafica media y desviación estándar móvil."""
    media_movil = serie.y.rolling(ventana, center=True).mean()
    std_movil = serie.y.rolling(ventana, center=True).std()

    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=[
            f"Serie + Media Móvil (ventana={ventana})",
            "Desv. Estándar Móvil — cambios indican heteroscedasticidad"
        ],
        vertical_spacing=0.15
    )

    fig.add_trace(go.Scatter(
        x=serie.ds, y=serie.y, mode="lines", name="Original",
        line=dict(color="lightblue", width=1.5)
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=serie.ds, y=media_movil, mode="lines", name="Media móvil",
        line=dict(color="#2196F3", width=2.5)
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=serie.ds, y=std_movil, mode="lines", name="Desv. estándar",
        line=dict(color="#FF5722", width=2), fill="tozeroy",
        fillcolor="rgba(255,87,34,0.1)"
    ), row=2, col=1)

    fig.update_layout(
        height=550, template="plotly_white",
        title="Detección de Cambios de Régimen — Estadísticas Móviles"
    )
    fig.show()


def evaluar_cambio_regimen(serie):
    """Compara dos subperíodos usando Levene y t-test."""
    n_total = len(serie)
    mitad = n_total // 2
    y1 = serie.y.values[:mitad]
    y2 = serie.y.values[mitad:]

    stat_var, p_var = stats.levene(y1, y2)
    stat_med, p_med = stats.ttest_ind(y1, y2)

    print("  TEST DE CAMBIO DE RÉGIMEN — Comparación de dos subperíodos")
    print(f"  Período 1: {serie.ds.iloc[0].strftime('%Y-%m')} → {serie.ds.iloc[mitad - 1].strftime('%Y-%m')}")
    print(f"  Período 2: {serie.ds.iloc[mitad].strftime('%Y-%m')} → {serie.ds.iloc[-1].strftime('%Y-%m')}")
    print()
    print(f"  Media P1={y1.mean():.0f}  |  Media P2={y2.mean():.0f}")
    print(f"  Desv. P1={y1.std():.0f}   |  Desv. P2={y2.std():.0f}")
    print()
    print(f"  Test Levene (varianzas iguales): stat={stat_var:.3f}  p={p_var:.4f}",
          "→ Varianzas distintas ⚠️" if p_var < 0.05 else "→ Varianzas similares ✅")
    print(f"  Test t (medias iguales):         stat={stat_med:.3f}  p={p_med:.4f}",
          "→ Medias distintas ⚠️" if p_med < 0.05 else "→ Medias similares ✅")
    print()
    print("  Interpretación:")
    if p_var < 0.05 or p_med < 0.05:
        print("  ⚠️  Hay evidencia de cambio estructural entre períodos.")
        print("     Estrategias: dummy variable, segmentar la serie, o modelar con SARIMA.")
    else:
        print("  ✅  Los dos subperíodos son estadísticamente similares.")
        print("     La serie no muestra cambios de régimen significativos.")

    return {"p_varianza": p_var, "p_media": p_med}

In [5]:
def crear_statsforecast_baseline(frecuencia="W-MON", estacionalidad=ESTACIONALIDAD_CUATRIENAL):
    """Crea el objeto StatsForecast con los modelos baseline del taller."""
    return StatsForecast(
        models=[
            Naive(),
            SeasonalNaive(season_length=estacionalidad),
            WindowAverage(window_size=3),
            RandomWalkWithDrift()
        ],
        freq=frecuencia
    )


def entrenar_y_predecir_baselines(train, test, frecuencia="W-MON", estacionalidad=ESTACIONALIDAD_CUATRIENAL):
    """Entrena los baselines y predice el horizonte del test."""
    horizonte = len(test)
    sf = crear_statsforecast_baseline(frecuencia=frecuencia, estacionalidad=estacionalidad)
    sf.fit(train)
    preds = sf.predict(h=horizonte)

    print("Pronósticos generados:")
    print(f"  Modelos: {[c for c in preds.columns if c not in ['unique_id', 'ds']]}")
    print(f"  Horizonte: {horizonte} semanas")
    display(preds.head())
    return sf, preds


def unir_predicciones_con_test(test, preds):
    """Une los valores reales del test con las predicciones."""
    return test.merge(preds, on=["unique_id", "ds"])


def graficar_pronosticos_baseline(serie, test, test_preds, fecha_corte, modelos_col):
    """Grafica histórico, test y pronósticos de cada baseline."""
    colores = ["#FF5722", "#4CAF50", "#9C27B0", "#FF9800"]
    fecha_corte = pd.to_datetime(fecha_corte)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=serie.ds, y=serie.y, mode="lines", name="Histórico",
        line=dict(color="lightgray", width=1)
    ))
    fig.add_trace(go.Scatter(
        x=test.ds, y=test.y, mode="lines+markers", name="Real (test)",
        line=dict(color="black", width=1), marker=dict(size=2)
    ))

    for modelo, color in zip(modelos_col, colores):
        fig.add_trace(go.Scatter(
            x=test_preds.ds, y=test_preds[modelo], mode="lines+markers", name=modelo,
            line=dict(color=color, width=1, dash="dot"), marker=dict(size=2)
        ))

    fig.add_shape(
        type="line", xref="x", yref="paper",
        x0=fecha_corte, x1=fecha_corte, y0=0, y1=1,
        line=dict(dash="dash", color="gray", width=1)
    )
    fig.add_annotation(
        x=fecha_corte, y=1.02, yref="paper",
        text="Inicio test", showarrow=False,
        font=dict(size=10, color="gray"), xanchor="left"
    )
    fig.update_layout(
        title="Comparación de Baselines — Casos Dengue",
        xaxis_title="Fecha", yaxis_title="Casos",
        height=480, template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02)
    )
    fig.show()


def calcular_metricas_baseline(train, test_preds, modelos_col):
    """Calcula MAE, RMSE, sMAPE y MASE para los modelos baseline."""
    naive_train_mae = float(np.abs(np.diff(train.y.values)).mean())

    evaluacion = evaluate(
        test_preds,
        metrics=[mae, rmse, smape],
        models=modelos_col,
        target_col="y"
    )

    fila_mae = evaluacion[evaluacion.metric == "mae"].copy()
    fila_mase = fila_mae.copy()
    fila_mase["metric"] = "mase"

    for modelo in modelos_col:
        fila_mase[modelo] = fila_mase[modelo] / naive_train_mae

    evaluacion = pd.concat([evaluacion, fila_mase], ignore_index=True)

    print(f"\nMAE del Naive sobre train (denominador MASE): {naive_train_mae:.2f}")
    print("=" * 60)
    print("  RESULTADOS POR MÉTRICA — MODELOS BASELINE")
    print("=" * 60)
    display(evaluacion)

    print("\n📌 Mejor modelo por métrica:")
    for metrica in ["mae", "rmse", "smape", "mase"]:
        fila = evaluacion[evaluacion.metric == metrica]
        mejor_idx = fila[modelos_col].values.argmin()
        mejor_modelo = modelos_col[mejor_idx]
        valor = fila[modelos_col].values[0][mejor_idx]
        print(f"   {metrica.upper():>6}: {mejor_modelo}  ({valor:.4f})")

    return evaluacion


def seleccionar_mejor_modelo(evaluacion, modelos_col, metrica="mae"):
    """Selecciona automáticamente el mejor modelo según una métrica."""
    fila = evaluacion[evaluacion.metric == metrica]
    valores = fila[modelos_col].values[0]
    mejor_idx = int(np.argmin(valores))
    return modelos_col[mejor_idx]


def graficar_metricas(evaluacion, modelos_col):
    """Grafica MAE, RMSE y sMAPE para comparar modelos."""
    fig = make_subplots(rows=1, cols=4, subplot_titles=["MAE", "MASE", "RMSE", "sMAPE"])
    colores_m = ["#2196F3", "#4CAF50", "#FF5722", "#9C27B0"]

    for col_idx, metrica in enumerate(["mae", "mase", "rmse", "smape"], start=1):
        fila = evaluacion[evaluacion.metric == metrica]
        vals = [float(fila[m].values[0]) for m in modelos_col]
        mejor_idx = int(np.argmin(vals))
        bar_colors = ["gold" if i == mejor_idx else colores_m[i] for i in range(len(modelos_col))]

        fig.add_trace(
            go.Bar(
                x=modelos_col,
                y=vals,
                marker_color=bar_colors,
                name=metrica,
                showlegend=False,
                text=[f"{v:.4f}" if metrica == "smape" else f"{v:.1f}" for v in vals],
                textposition="outside"
            ),
            row=1,
            col=col_idx
        )

    fig.update_layout(
        height=400, template="plotly_white",
        title="Comparación de Métricas por Modelo<br><sup>Barra dorada = mejor modelo en esa métrica</sup>"
    )
    fig.show()

In [6]:
def graficar_acf_plotly(serie, titulo, n_lags=30, color="#2196F3"):
    """ACF interactivo con Plotly. Barras rojas = significativas al 95%."""
    arr = np.asarray(serie).astype(float)
    arr = arr[~np.isnan(arr)]
    acf_vals = acf(arr, nlags=n_lags, fft=True)
    ci = 1.96 / np.sqrt(len(arr))
    lags = np.arange(len(acf_vals))

    fig = go.Figure()
    for lag in lags:
        bar_color = color if abs(acf_vals[lag]) <= ci else "crimson"
        fig.add_trace(go.Scatter(
            x=[lag, lag], y=[0, acf_vals[lag]], mode="lines",
            line=dict(color=bar_color, width=2.5), showlegend=False
        ))

    fig.add_trace(go.Scatter(
        x=lags, y=acf_vals, mode="markers", showlegend=False,
        marker=dict(color=[color if abs(v) <= ci else "crimson" for v in acf_vals], size=6)
    ))
    fig.add_hline(y=ci, line_dash="dash", line_color="gray", opacity=0.7, annotation_text=f"IC 95% = ±{ci:.3f}")
    fig.add_hline(y=-ci, line_dash="dash", line_color="gray", opacity=0.7)
    fig.add_hline(y=0, line_color="black", line_width=0.8)
    fig.update_layout(
        title=titulo,
        xaxis_title="Lag",
        yaxis_title="Autocorrelación",
        height=360,
        template="plotly_white",
        yaxis=dict(range=[-1.05, 1.05])
    )
    return fig


def test_estacionaridad(serie, nombre):
    """
    Aplica ADF + KPSS y entrega una conclusión conjunta.
    ADF p<0.05 + KPSS p>=0.05 → estacionaria.
    """
    arr = np.asarray(serie).astype(float)
    arr = arr[~np.isnan(arr)]

    adf_stat, adf_p, _, _, _, _ = adfuller(arr, autolag="AIC")
    kpss_stat, kpss_p, _, _ = kpss(arr, regression="c", nlags="auto")

    adf_est = adf_p < 0.05
    kpss_est = kpss_p >= 0.05

    if adf_est and kpss_est:
        conclusion = "✅  ESTACIONARIA"
    elif not adf_est and not kpss_est:
        conclusion = "❌  NO ESTACIONARIA"
    elif adf_est and not kpss_est:
        conclusion = "⚠️  INCIERTA (posible tendencia)"
    else:
        conclusion = "⚠️  INCIERTA (cerca del límite)"

    print(f"\n{'═' * 58}")
    print(f"  {nombre}")
    print(f"{'═' * 58}")
    print(f"  ADF:  stat={adf_stat:8.4f}  p={adf_p:.4f}",
          "→ ES estacionaria ✅" if adf_est else "→ NO estacionaria ❌")
    print(f"  KPSS: stat={kpss_stat:8.4f}  p={kpss_p:.4f}",
          "→ ES estacionaria ✅" if kpss_est else "→ NO estacionaria ❌")
    print(f"  {'─' * 54}")
    print(f"  CONCLUSIÓN: {conclusion}")

    return {"adf_p": adf_p, "kpss_p": kpss_p, "estacionaria": adf_est and kpss_est}


def prueba_ljungbox(serie, lags_test=None, titulo="TEST DE LJUNG-BOX"):
    """Aplica Ljung-Box para varios rezagos."""
    if lags_test is None:
        lags_test = [1, 6, 12, 18, 24]

    valores = np.asarray(serie).astype(float)
    valores = valores[~np.isnan(valores)]

    print(titulo)
    print("H₀: ρ₁ = ρ₂ = ··· = ρₕ = 0  (no hay autocorrelación)")
    print("─" * 62)
    print(f"{'Lags':>6}  {'Estadístico Q':>14}  {'p-valor':>10}  {'Conclusión'}")
    print("─" * 62)

    resultados = []
    for lag in lags_test:
        result = acorr_ljungbox(valores, lags=[lag], return_df=True)
        q_stat = result["lb_stat"].iloc[0]
        p_val = result["lb_pvalue"].iloc[0]
        conc = "Rechaza H₀ — HAY autocorrelación ❌" if p_val < 0.05 else "No rechaza H₀ — sin autocorrelación ✅"
        print(f"  {lag:>4}  {q_stat:>14.4f}  {p_val:>10.6f}  {conc}")
        resultados.append({"lag": lag, "q_stat": q_stat, "p_valor": p_val})

    print("─" * 62)
    return pd.DataFrame(resultados)


def graficar_ljungbox_pvalores(serie, max_lag=30):
    """Grafica p-valores de Ljung-Box por rezago."""
    valores = np.asarray(serie).astype(float)
    valores = valores[~np.isnan(valores)]
    lags_all = list(range(1, max_lag + 1))
    results_lb = acorr_ljungbox(valores, lags=lags_all, return_df=True)
    p_valores = results_lb["lb_pvalue"].values

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=lags_all,
        y=p_valores,
        mode="lines+markers",
        line=dict(color="#2196F3", width=2),
        marker=dict(color=["red" if p < 0.05 else "green" for p in p_valores], size=7),
        name="p-valor Ljung-Box"
    ))
    fig.add_hline(y=0.05, line_dash="dash", line_color="red", annotation_text="α = 0.05", annotation_position="right")
    fig.update_layout(
        title="Test de Ljung-Box — p-valores por lag<br><sup>Puntos rojos = evidencia de autocorrelación significativa</sup>",
        xaxis_title="Lag",
        yaxis_title="p-valor",
        height=360,
        template="plotly_white",
        yaxis=dict(range=[-0.05, 1.05])
    )
    fig.show()


def diagnosticar_residuos(test_preds, mejor_modelo, lags_residuos=None):
    """Calcula residuos del mejor modelo y aplica ACF + Ljung-Box."""
    if lags_residuos is None:
        lags_residuos = [1, 6, 12]

    residuos = test_preds["y"] - test_preds[mejor_modelo]
    residuos_arr = residuos.values

    print(f"Residuos del modelo {mejor_modelo}")
    print(f"Media: {residuos_arr.mean():.2f}  |  Desviación estándar: {residuos_arr.std():.2f}")
    print("Esperado en ruido blanco: media ≈ 0 y desviación estándar relativamente constante")

    print(f"\nLJUNG-BOX SOBRE RESIDUOS DEL {mejor_modelo}:")
    print("─" * 58)
    for lag in lags_residuos:
        result = acorr_ljungbox(residuos_arr, lags=[lag], return_df=True)
        p_val = result["lb_pvalue"].iloc[0]
        conc = "Quedan patrones ⚠️  → margen de mejora" if p_val < 0.05 else "Residuos ≈ ruido blanco ✅"
        print(f"  Lag {lag:>2}: p={p_val:.4f}  →  {conc}")

    acf_resid = acf(residuos_arr, nlags=min(52, len(residuos_arr) - 1), fft=True)
    ci_r = 1.96 / np.sqrt(len(residuos_arr))

    fig = make_subplots(
        rows=1,
        cols=3,
        subplot_titles=["Residuos en el tiempo", "Distribución", "ACF de residuos"],
        horizontal_spacing=0.08
    )
    fig.add_trace(go.Scatter(
        x=test_preds.ds, y=residuos, mode="lines+markers",
        line=dict(color="#FF5722", width=1.8), marker=dict(size=3), name="Residuos"
    ), row=1, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color="black", row=1, col=1)
    fig.add_hline(y=2 * residuos_arr.std(), line_dash="dot", line_color="gray", row=1, col=1)
    fig.add_hline(y=-2 * residuos_arr.std(), line_dash="dot", line_color="gray", row=1, col=1)

    fig.add_trace(go.Histogram(
        x=residuos,
        nbinsx=30,
        marker_color="#9C27B0",
        opacity=0.75,
        name="Distribución"
    ), row=1, col=2)
    fig.add_vline(x=0, line_dash="dash", line_color="black", row=1, col=2)

    for lag_r in range(len(acf_resid)):
        color_barra = "crimson" if abs(acf_resid[lag_r]) > ci_r else "#4CAF50"
        fig.add_trace(go.Scatter(
            x=[lag_r, lag_r], y=[0, acf_resid[lag_r]],
            mode="lines", line=dict(color=color_barra, width=2.5), showlegend=False
        ), row=1, col=3)

    fig.add_hline(y=ci_r, line_dash="dash", line_color="gray", row=1, col=3)
    fig.add_hline(y=-ci_r, line_dash="dash", line_color="gray", row=1, col=3)
    fig.update_layout(
        height=390,
        template="plotly_white",
        showlegend=False,
        title=f"Diagnóstico de Residuos — {mejor_modelo}<br><sup>Tiempo, distribución y autocorrelación remanente</sup>"
    )
    fig.update_xaxes(title_text="Fecha", row=1, col=1)
    fig.update_yaxes(title_text="Residuo", row=1, col=1)
    fig.update_xaxes(title_text="Residuo", row=1, col=2)
    fig.update_xaxes(title_text="Lag", row=1, col=3)
    fig.update_yaxes(title_text="ACF", row=1, col=3, range=[-1.05, 1.05])
    fig.show()
    return residuos


In [7]:
def crear_transformaciones(serie):
    """Agrega transformaciones útiles para diagnóstico: log1p, diferencia y diferencia logarítmica."""
    transformada = serie.copy()
    transformada["y_log"] = np.log1p(transformada["y"])
    transformada["y_diff"] = transformada["y"].diff()
    transformada["y_log_diff"] = transformada["y_log"].diff()
    return transformada


def graficar_transformaciones(serie_transformada):
    """Grafica serie original y transformaciones principales."""
    fig = make_subplots(
        rows=3, cols=1,
        subplot_titles=[
            "Serie original",
            "Transformación log1p(y)",
            "Diferencia logarítmica diff(log1p(y))"
        ],
        vertical_spacing=0.12
    )

    fig.add_trace(go.Scatter(
        x=serie_transformada.ds, y=serie_transformada.y,
        mode="lines", name="Original", line=dict(color="#2196F3", width=1.5)
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=serie_transformada.ds, y=serie_transformada.y_log,
        mode="lines", name="log1p(y)", line=dict(color="#4CAF50", width=1.5)
    ), row=2, col=1)

    fig.add_trace(go.Scatter(
        x=serie_transformada.ds, y=serie_transformada.y_log_diff,
        mode="lines", name="diff(log1p(y))", line=dict(color="#FF5722", width=1.5)
    ), row=3, col=1)

    fig.update_layout(
        height=720,
        template="plotly_white",
        title="Transformaciones de la Serie Temporal"
    )
    fig.show()


def ejecutar_validacion_cruzada(sf, serie, h=52, n_windows=3, step_size=52):
    """Ejecuta cross-validation temporal con StatsForecast."""
    cv_results = sf.cross_validation(
        df=serie,
        h=h,
        n_windows=n_windows,
        step_size=step_size
    )

    print(f"Resultados CV: {cv_results.shape[0]} filas × {cv_results.shape[1]} columnas")
    print(f"Folds (cutoffs): {cv_results.cutoff.unique().tolist()}")
    display(cv_results.head(6))
    return cv_results


def evaluar_validacion_cruzada(cv_results, modelos_col):
    """Calcula métricas promedio en la validación cruzada temporal."""
    cv_models = [c for c in modelos_col if c in cv_results.columns]
    cv_eval = evaluate(
        cv_results,
        metrics=[mae, rmse, smape],
        models=cv_models,
        target_col="y"
    )

    print("Métricas promedio en Cross-Validation:")
    print(cv_eval.to_string(index=False))
    return cv_eval


def graficar_validacion_cruzada(serie, cv_results, modelo="SeasonalNaive"):
    """Grafica los pronósticos de validación cruzada para un modelo seleccionado."""
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=serie.ds,
        y=serie.y,
        mode="lines",
        name="Serie real",
        line=dict(color="black", width=1.5)
    ))

    colores_cv = ["#FF5722", "#4CAF50", "#9C27B0", "#FF9800", "#2196F3"]
    for i, cutoff in enumerate(sorted(cv_results.cutoff.unique())):
        fold = cv_results[cv_results.cutoff == cutoff]
        color = colores_cv[i % len(colores_cv)]
        fig.add_trace(go.Scatter(
            x=fold.ds,
            y=fold[modelo],
            mode="lines+markers",
            name=f"{modelo} Fold {i + 1} (cutoff {cutoff.strftime('%Y-%m')})",
            line=dict(color=color, width=3, dash="dot"),
            marker=dict(size=2)
        ))
        fig.add_shape(
            type="line", xref="x", yref="paper",
            x0=cutoff, x1=cutoff, y0=0, y1=1,
            line=dict(dash="dash", color=color, width=1.5),
            opacity=0.4
        )

    fig.update_layout(
        title=f"Time Series Cross-Validation — {modelo}",
        xaxis_title="Fecha",
        yaxis_title="Casos",
        height=420,
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02)
    )
    fig.show()

---
## 📦 PARTE 2: Carga de Datos y Formato Nixtla

Los datos originales contienen reportes individuales de casos de dengue. Para trabajar como serie temporal, se agrupan por semana epidemiológica y se llevan al formato requerido por Nixtla:

```text
unique_id | ds | y
```

Donde `ds` es la fecha de la semana y `y` es el número de casos observados.

### Criterio de construcción de la serie

La agregación semanal es una decisión adecuada para dengue porque reduce ruido diario, se alinea con la lógica epidemiológica de vigilancia y permite comparar períodos de transmisión de forma más estable. Al mismo tiempo, conserva suficiente granularidad para detectar aceleraciones tempranas de brotes.

El uso del formato Nixtla (`unique_id`, `ds`, `y`) no es solo una exigencia de librería. También impone una estructura clara: una unidad de análisis, una marca temporal ordenada y una variable objetivo. Esta claridad facilita reproducibilidad, validación cruzada temporal y comparación homogénea entre modelos.


In [8]:
datos_crudos = leer_datos_dengue(RUTA_DATOS_CRUDOS)

df = construir_serie_semanal_nixtla(
    datos=datos_crudos,
    columna_fecha=COLUMNA_FECHA,
    unique_id=UNIQUE_ID
)

guardar_serie_nixtla(df, RUTA_SALIDA_NIXTLA)

print("\nSerie semanal en formato Nixtla:")
display(df.head(20))
print("\nÚltimas observaciones:")
display(df.tail())

Primeras filas del dataset original:


,fec_not,semana
0,2010-12-03,48
1,2010-02-25,7
2,2010-01-16,1
3,2010-05-24,19
4,2010-03-26,11



Columnas disponibles:
['fec_not', 'semana']
Archivo generado correctamente:
dengue_anio_semana_nixtla.csv

Serie semanal en formato Nixtla:


,unique_id,ds,y
0,dengue_cali,2009-12-28,12
1,dengue_cali,2010-01-04,142
2,dengue_cali,2010-01-11,210
3,dengue_cali,2010-01-18,253
4,dengue_cali,2010-01-25,345
5,dengue_cali,2010-02-01,375
6,dengue_cali,2010-02-08,504
7,dengue_cali,2010-02-15,573
8,dengue_cali,2010-02-22,543
9,dengue_cali,2010-03-01,475



Últimas observaciones:


,unique_id,ds,y
736,dengue_cali,2024-02-05,4
737,dengue_cali,2024-02-12,7
738,dengue_cali,2024-02-19,6
739,dengue_cali,2024-03-04,5
740,dengue_cali,2024-03-25,1


### 📈 Visualización inicial de la serie

Esta gráfica permite observar el comportamiento general de los casos de dengue a lo largo del tiempo: picos, períodos de mayor transmisión, posibles cambios de nivel y ciclos recurrentes.

In [9]:
graficar_serie(df, titulo="Número de Casos de Dengue por Semana")

### 🔍 ¿Qué observar?

- **Picos epidémicos:** semanas con aumentos fuertes de casos.
- **Persistencia temporal:** cuando los casos altos tienden a mantenerse durante varias semanas.
- **Cambios de régimen:** períodos donde la media o la variabilidad parecen cambiar.
- **Estacionalidad:** repetición de patrones en ciertos momentos del año, cada 3-4.

### Análisis preliminar de la serie

1. Brotes epidémicos recurrentes. Se observan varios picos grandes de casos:
2010, 2013 a 2014, 2016, 2020, 2023 a 2024.

Esto indica ciclos epidémicos recurrentes muy típicos del dengue.

2. Estacionalidad evidente. Los brotes no ocurren aleatoriamente.
Parecen repetirse en ciertos períodos del tiempo. Esto sugiere fuerte influencia de otros factores como lluvias, temperatura, humedad, dinámica vectorial.
Es decir se observa un componente estacional importante.

3. Cambios de régimen
La serie alterna entre periodos endémicos con pocos casos y periodos epidémicos
con crecimiento explosivo.
Eso indica comportamiento multi-régimen muy típico de sistemas infecciosos complejos.

4. No linealidad Los aumentos no son graduales.
Los brotes aparecen abruptamente, aceleradamente, de forma explosiva.

Esto nos indica dinámica no lineal y posible comportamiento aleatorio o altamente sensible a condiciones externas.

5. Heteroscedasticidad:
La variabilidad cambia mucho en el tiempo.
En años tranquilos poca dispersión, estabilidad.
En epidemias se observa gran volatilidad y cambios extremos.

6. Persistencia temporal:
Los brotes duran varias semanas. No son picos aislados.
Esto se relaciona con memoria temporal epidemiológica coherente con transmisión infecciosa.

7. Ruptura estructural reciente (2023–2024):
El brote más reciente parece más intenso que varios anteriores y además crece rápidamente, alcanza niveles muy altos. Esto podría indicar nuevo comportamiento epidemiológico o cambio estructural.

8. Posibles problemas al final de la serie
La caída abrupta cercana a 2024 podría deberse a retraso de reporte, truncamiento de datos, semanas incompletas.



### Mapa de calor año-semana

Esta vista resume la intensidad de casos por año y semana epidemiológica. Es útil para identificar brotes, comparar años críticos y revisar si los picos se concentran en semanas similares.


In [10]:
graficar_heatmap_anio_semana(df)


---
## 🔍 PARTE 3: Calidad de Datos

Antes de cualquier modelado, se revisa la continuidad temporal de la serie. En esta versión, primero se diagnostican faltantes y luego se completa el calendario semanal. La imputación se calcula después del split temporal para evitar usar información del test durante el entrenamiento.

### 3.1 Valores Faltantes — Detección

La primera validación consiste en comprobar si el calendario semanal está completo. Este paso es fundamental porque muchos modelos de series temporales asumen observaciones igualmente espaciadas. Si faltan semanas y no se corrigen, el modelo puede confundir ausencia de registros con ausencia de casos, o interpretar saltos de calendario como cambios reales en la transmisión.

En este análisis, la imputación se plantea después del split temporal. Esta decisión evita fuga de información: las reglas usadas para completar el conjunto de entrenamiento no deben aprender nada del período de prueba. Aunque parezca un detalle operativo, es una condición básica para que la evaluación fuera de muestra sea creíble.


In [11]:
fechas_faltantes = diagnosticar_faltantes(df, frecuencia=FRECUENCIA)

df = completar_calendario_semanal(
    serie=df,
    frecuencia=FRECUENCIA,
    unique_id=UNIQUE_ID
)

print("\nDespués de completar el calendario semanal:")
diagnosticar_faltantes(df, frecuencia=FRECUENCIA)

  DIAGNÓSTICO DE VALORES FALTANTES
  Observaciones esperadas: 744
  Observaciones presentes: 741
  Fechas faltantes:        3
  NaN en columna y:        0

  ⚠️  Fechas faltantes: [Timestamp('2024-02-26 00:00:00'), Timestamp('2024-03-11 00:00:00'), Timestamp('2024-03-18 00:00:00')]

Después de completar el calendario semanal:
  DIAGNÓSTICO DE VALORES FALTANTES
  Observaciones esperadas: 744
  Observaciones presentes: 744
  Fechas faltantes:        0
  NaN en columna y:        3

  ⚠️  Fechas faltantes: []


DatetimeIndex([], dtype='datetime64[ns]', freq='W-MON')

---
## ✂️ PARTE 4: Train-Test Split Temporal

**Regla crítica:** nunca se aleatoriza una serie temporal. El modelo debe aprender del pasado para predecir el futuro.

Se hace el corte antes de calcular las medias de imputación, de forma que cualquier regla aprendida para completar datos provenga únicamente del conjunto de entrenamiento.

### Justificación del corte temporal

El corte en 2020 deja un conjunto de entrenamiento amplio y un conjunto de prueba suficientemente desafiante. El entrenamiento cubre aproximadamente diez años de historia, incluyendo ciclos epidémicos relevantes, el test contiene un período reciente con fuerte variabilidad lo que permite evaluar si los baselines generalizan fuera de la ventana histórica inicial.

Esta separación es más exigente que una partición aleatoria, pero es la única coherente para este tipo de pronósticos. Mezclar semanas futuras dentro del entrenamiento produciría una evaluación optimista y metodológicamente inválida.


In [12]:
train, test = separar_train_test(df, FECHA_CORTE)

print(f"TRAIN: {len(train)} observaciones  ({train.ds.min().strftime('%Y-%m')} → {train.ds.max().strftime('%Y-%m')})")
print(f"TEST:  {len(test)} observaciones  ({test.ds.min().strftime('%Y-%m')} → {test.ds.max().strftime('%Y-%m')})")
print(f"Proporción: {len(train) / len(df) * 100:.0f}% / {len(test) / len(df) * 100:.0f}%")

graficar_split(train, test, FECHA_CORTE)

TRAIN: 523 observaciones  (2009-12 → 2019-12)
TEST:  221 observaciones  (2020-01 → 2024-03)
Proporción: 70% / 30%


### 4.1 Imputación por Media Estacional

Dado que la serie puede presentar estacionalidad semanal/anual, se utiliza la media por semana del año. Para evitar fuga de información, las medias estacionales se calculan **solo con train** y luego se aplican a `train` y `test`.

In [13]:
medias_estacionales, media_global_train = calcular_medias_estacionales(train)

train = imputar_por_media_estacional(train, medias_estacionales, media_global_train)
test = imputar_por_media_estacional(test, medias_estacionales, media_global_train)

df = pd.concat([train, test], ignore_index=True).sort_values("ds").reset_index(drop=True)

print(f"NaN en train después de imputación: {train.y.isna().sum()}")
print(f"NaN en test después de imputación:  {test.y.isna().sum()}")
print(f"NaN en df después de imputación:    {df.y.isna().sum()}")

display(df.head())
display(df.tail())

NaN en train después de imputación: 0
NaN en test después de imputación:  0
NaN en df después de imputación:    0


,unique_id,ds,y
0,dengue_cali,2009-12-28,12.0
1,dengue_cali,2010-01-04,142.0
2,dengue_cali,2010-01-11,210.0
3,dengue_cali,2010-01-18,253.0
4,dengue_cali,2010-01-25,345.0


,unique_id,ds,y
739,dengue_cali,2024-02-26,121.8
740,dengue_cali,2024-03-04,5.0
741,dengue_cali,2024-03-11,103.2
742,dengue_cali,2024-03-18,89.8
743,dengue_cali,2024-03-25,1.0


---
## 🔎 PARTE 5: Outliers y Anomalías

En una serie epidemiológica, los valores extremos no deben eliminarse automáticamente. Un pico puede ser un error de datos, pero también puede representar un brote real. Por eso, esta sección se usa como diagnóstico, no como criterio directo de eliminación.

### 5.1 Detección de Outliers — IQR y Z-score

In [14]:
outliers_iqr, q1, q3, iqr, limite_inf_iqr, limite_sup_iqr = detectar_outliers_iqr(df)
outliers_z = detectar_outliers_zscore(df, umbral=3)

print("IQR — Detección de Outliers")
print(f"  Q1={q1:.0f}  Q3={q3:.0f}  IQR={iqr:.0f}")
print(f"  Límite inferior: {limite_inf_iqr:.0f}")
print(f"  Límite superior: {limite_sup_iqr:.0f}")
print(f"  Outliers detectados: {len(outliers_iqr)}")
display(outliers_iqr[["ds", "y"]])

print(f"\nZ-score (|z| > 3) — Outliers detectados: {len(outliers_z)}")
if len(outliers_z):
    display(outliers_z[["ds", "y"]])

graficar_outliers_iqr(df, outliers_iqr, limite_inf_iqr, limite_sup_iqr)

IQR — Detección de Outliers
  Q1=17  Q3=107  IQR=90
  Límite inferior: -118
  Límite superior: 242
  Outliers detectados: 57


,ds,y
3,2010-01-18,253.0
4,2010-01-25,345.0
5,2010-02-01,375.0
6,2010-02-08,504.0
7,2010-02-15,573.0
8,2010-02-22,543.0
9,2010-03-01,475.0
10,2010-03-08,451.0
11,2010-03-15,439.0
12,2010-03-22,384.0



Z-score (|z| > 3) — Outliers detectados: 17


,ds,y
6,2010-02-08,504.0
7,2010-02-15,573.0
8,2010-02-22,543.0
9,2010-03-01,475.0
10,2010-03-08,451.0
11,2010-03-15,439.0
12,2010-03-22,384.0
526,2020-01-27,438.0
530,2020-02-24,379.0
532,2020-03-09,396.0


### Interpretación de Outliers

Los outliers detectados por IQR, en el caso del dengue, pueden representar brotes reales o semanas epidemiológicas con transmisión inusualmente alta. Por esta razón, se recomienda mantenerlos en la serie, salvo que exista evidencia externa de error de registro.

Con los datos actuales, el criterio IQR global identifica alrededor de **57 semanas extremas**, mientras que un criterio tipo Z-score es más restrictivo.

La decisión analítica recomendada es **no eliminar ni winsorizar** estos valores en la fase exploratoria. En epidemiología, los extremos son parte del fenómeno de interés y son justamente las semanas que un sistema de alerta temprana debería anticipar, eliminarlos podría producir métricas aparentemente mejores pero a costa de ocultar la dinámica más importante desde el punto de vista sanitario.

###Brotes epidémicos
Los puntos rojos aparecen concentrados en los picos de la serie:
2010
2013
2016
2020
2023 - 2024
Esto indica que representan semanas de transmisión excepcionalmente alta, es decir brotes epidémicos severos y no simples anomalías estadísticas.
En dengue esto es completamente esperado.
Estos outliers suelen estar asociados a cambios climáticos extremos, fenómeno de El Niño / La Niña, aumento de lluvias, aumento de temperatura, aparición de nuevos serotipos, disminución de inmunidad poblacional, fallas en campañas de control vectorial, saturación hospitalaria, subregistro previo seguido de corrección.
Es decir que representan cambios reales del proceso epidemiológico, no errores de medición.

###Alta concentración en ciertos años
Especialmente fuerte en 2010 y 2023 al 2024
Esto sugiere que esos años fueron años epidémicos extraordinarios y probablemente deberían analizarse por separado.
Pueden incluso justificar:
Segmentación del modelo.
Análisis por régimen epidemiológico.
Modelos con change points.

En conclusión es importante dejar los outliers para el futuro modelo predictivo.


In [15]:
df_check = detectar_outliers_iqr_por_mes(df)
n_out_mes = df_check.outlier_mensual.sum()

print(f"Outliers con IQR por mes: {n_out_mes}")
if n_out_mes:
    display(df_check[df_check.outlier_mensual][["ds", "y", "mes"]])
else:
    print("✅ Sin outliers cuando se aplica IQR por estación")

Outliers con IQR por mes: 56


,ds,y,mes
3,2010-01-18,253.0,1
4,2010-01-25,345.0,1
5,2010-02-01,375.0,2
6,2010-02-08,504.0,2
7,2010-02-15,573.0,2
8,2010-02-22,543.0,2
9,2010-03-01,475.0,3
10,2010-03-08,451.0,3
11,2010-03-15,439.0,3
12,2010-03-22,384.0,3


### 5.2 Cambios de Régimen y Anomalías Estructurales

Un cambio de régimen ocurre cuando las propiedades estadísticas de la serie cambian de forma importante: por ejemplo, la media, la varianza o la intensidad de los brotes.

In [16]:
graficar_cambios_regimen(df, ventana=2)

In [17]:
resultado_regimen = evaluar_cambio_regimen(df)

  TEST DE CAMBIO DE RÉGIMEN — Comparación de dos subperíodos
  Período 1: 2009-12 → 2017-02
  Período 2: 2017-02 → 2024-03

  Media P1=96  |  Media P2=71
  Desv. P1=96   |  Desv. P2=97

  Test Levene (varianzas iguales): stat=2.847  p=0.0920 → Varianzas similares ✅
  Test t (medias iguales):         stat=3.447  p=0.0006 → Medias distintas ⚠️

  Interpretación:
  ⚠️  Hay evidencia de cambio estructural entre períodos.
     Estrategias: dummy variable, segmentar la serie, o modelar con SARIMA.


### Interpretación de Cambios de Régimen

Se observan diferencias importantes entre subperíodos, esto sugiere que la serie no se comporta igual durante todo el horizonte histórico. En dengue esto obedece a ciclos epidémicos, cambios climáticos, variaciones en vigilancia epidemiológica, intervenciones de salud pública, movilidad poblacional, etc.

Al dividir la serie en dos mitades, el primer subperíodo muestra una media semanal aproximada de **96 casos**, mientras que el segundo se ubica cerca de **71 casos**. Sin embargo, la desviación estándar se mantiene alta en ambos tramos, alrededor de **95-97 casos**. Esto sugiere que no solo cambia el nivel promedio, sino que la serie conserva una volatilidad considerable incluso cuando la transmisión media es baja.

Se observan claramente varios cambios de régimen, los cuales se podrían clasificar de la siguiente manera:

Régimen de alta transmisión:  Cuando la media móvil sube fuertemente los periodos son los siguientes: 2010, 2013 al 2014, 2016, 2020, 2023 al 2024, estos corresponden a brotes epidémicos.

Régimen de baja transmisión: Cuando la media móvil permanece cerca de cero, en este caso, los periodos son los siguientes: 2011 al 2012, 2017 al 2019. Esto representa períodos endémicos controlados con baja circulación viral.

Adicionalmente se realizaron los test de Lavene y T, evidenciando un cambio estructural en los períodos:



### 5.3 Descomposición STL

La descomposición STL separa la serie en tendencia, estacionalidad y residuo. Aquí se usa un período cuatrienal de 208 semanas para revisar si la dinámica está dominada por ciclos estacionales multianuales o por brotes y cambios de nivel.


In [18]:
stl_resultado = graficar_descomposicion_stl(df, periodo=ESTACIONALIDAD_CUATRIENAL)


---
## 📈 PARTE 6: Análisis de Autocorrelación de la Serie

La ACF mide cuánto se parece la serie a sí misma en distintos rezagos. En series epidemiológicas, una autocorrelación alta suele indicar persistencia temporal: semanas con muchos casos tienden a estar cerca de otras semanas con muchos casos.

In [19]:
fig_acf_208 = graficar_acf_plotly(
    df.y,
    "ACF - Casos de Dengue en la ciudad de Cali",
    n_lags=ESTACIONALIDAD_CUATRIENAL
)
fig_acf_208.show()

In [20]:
fig_acf_52 = graficar_acf_plotly(
    df.y,
    "ACF - Casos de Dengue en la ciudad de Cali (zoom 52 semanas)",
    n_lags=52
)
fig_acf_52.show()

### Observaciones ACF

La ACF hasta 52 semanas muestra una autocorrelación de corto plazo muy alta, con valores cercanos a **0.95** en el lag 1, **0.83** en el lag 4 y **0.64** en el lag 8. Esto confirma una fuerte persistencia temporal: los brotes no aparecen como eventos aislados, sino que crecen y decrecen durante varias semanas. A medida que aumentan los rezagos, la autocorrelación cae de forma clara, y alrededor del lag 52 no se observa una señal anual fuerte. En términos epidemiológicos, es coherente con brotes que crecen y decrecen gradualmente, no con eventos independientes semana a semana.

Esta lectura es compatible con las métricas: `WindowAverage` compite bien en `RMSE` y `sMAPE` porque aprovecha la información reciente, mientras que SeasonalNaive puede ser útil en `MAE`, pero la ACF no muestra una estacionalidad anual o cuatrienal suficientemente clara como para depender solo de un patrón estacional fijo.

Se observa que los primeros lags tienen valores muy cercanos a 1. Esto significa que los casos actuales dependen fuertemente del pasado reciente

Es decir, si esta semana hay muchos casos,la siguiente semana probablemente también.
Este comportamiento es esperado en dengue.
Se observa un decaimiento lento. La autocorrelación disminuye lentamente no cae bruscamente permanece significativa hasta un lag de 25. Esto indica una fuerte persistencia temporal y también posible no estacionariedad, porque en series estacionarias la ACF suele caer mucho más rápido en este caso no ocurre eso.

No parece ruido blanco, si fuera ruido blanco ACF(k)≈0 para casi todos los lags.
Pero aquí casi todos los lags son significativos por encima del IC 95%, entonces la serie tiene estructura temporal real no es aleatoria.


---
## 📐 PARTE 7: Tests de Estacionariedad — ADF y KPSS

Los tests ADF y KPSS ayudan a evaluar si la serie es estacionaria. En la práctica, ambos se leen de forma complementaria:

- **ADF p < 0.05:** evidencia a favor de estacionariedad.
- **KPSS p >= 0.05:** evidencia compatible con estacionariedad.

In [21]:
r_orig = test_estacionaridad(df.y, "Casos de dengue en Cali — Serie Original")


══════════════════════════════════════════════════════════
  Casos de dengue en Cali — Serie Original
══════════════════════════════════════════════════════════
  ADF:  stat= -5.0818  p=0.0000 → ES estacionaria ✅
  KPSS: stat=  0.1358  p=0.1000 → ES estacionaria ✅
  ──────────────────────────────────────────────────────
  CONCLUSIÓN: ✅  ESTACIONARIA


### Interpretación de Tests de Estacionariedad

La serie original entrega **ADF p ≈ 0.0** y **KPSS p ≈ 0,10**. Bajo una lectura estricta de estas pruebas, hay evidencia compatible con estacionariedad en la serie agregada semanal.

Esta conclusión estadística debe interpretarse con cautela. Aunque las pruebas favorecen estacionariedad, los gráficos muestran brotes, cambios de nivel y heterogeneidad de varianza. En dengue, la estacionariedad no implica estabilidad epidemiológica plena, más bien indica que para ciertos modelos la serie puede ser trabajable sin diferenciar obligatoriamente. La decisión final debe combinar tests, visualización, ACF y desempeño predictivo.


---
## 🔬 PARTE 8: Test de Ljung-Box sobre la Serie Original

El test de Ljung-Box evalúa si varias autocorrelaciones son simultáneamente cero. Sirve para confirmar si la serie tiene estructura temporal explotable por modelos de pronóstico.

In [22]:
resultados_ljungbox = prueba_ljungbox(
    df.y.values,
    lags_test=[1, 6, 12, 18, 24],
    titulo="TEST DE LJUNG-BOX — Casos de Dengue Cali"
)

TEST DE LJUNG-BOX — Casos de Dengue Cali
H₀: ρ₁ = ρ₂ = ··· = ρₕ = 0  (no hay autocorrelación)
──────────────────────────────────────────────────────────────
  Lags   Estadístico Q     p-valor  Conclusión
──────────────────────────────────────────────────────────────
     1        680.3438    0.000000  Rechaza H₀ — HAY autocorrelación ❌
     6       3264.0867    0.000000  Rechaza H₀ — HAY autocorrelación ❌
    12       4851.4210    0.000000  Rechaza H₀ — HAY autocorrelación ❌
    18       5562.6777    0.000000  Rechaza H₀ — HAY autocorrelación ❌
    24       5778.2538    0.000000  Rechaza H₀ — HAY autocorrelación ❌
──────────────────────────────────────────────────────────────


In [23]:
graficar_ljungbox_pvalores(df.y.values, max_lag=30)

### Observaciones Ljung-Box

Si los p-valores son menores a 0.05, se rechaza la hipótesis de ausencia de autocorrelación. En ese caso, la serie no se comporta como ruido blanco y tiene estructura temporal que puede ser aprovechada por modelos de pronóstico.

Con los datos actuales, Ljung-Box rechaza con claridad la hipótesis de ruido blanco: para todos los lags los p-valores son numéricamente indistinguibles de cero. Esto valida formalmente lo observado en la ACF: los casos semanales de dengue tienen dependencia temporal fuerte.

La conclusión estadística debe traducirse en una conclusión de negocio o salud pública: existe información útil en el pasado reciente de la serie. Por tanto, un sistema de pronóstico debe aprovechar rezagos, tendencia local, posibles ciclos estacionales y señales externas si están disponibles. Después de ajustar modelos, el mismo test debe repetirse sobre residuos para verificar si esa estructura temporal fue realmente absorbida.


---
## 🤖 PARTE 9: Baselines con `statsforecast`

Los modelos baseline se entrenan sobre la escala original de casos. Esto permite construir una referencia simple e interpretable antes de pasar a modelos más complejos o transformaciones.

Modelos utilizados:

- `Naive`: repite el último valor observado.
- `SeasonalNaive`: usa la observación de hace 208 semanas.
- `WindowAverage`: promedio de las últimas observaciones.
- `RandomWalkWithDrift`: caminata aleatoria con tendencia.

Se evaluaron cuatro modelos baseline con statsforecast: Naive, SeasonalNaive, WindowAverage y RandomWalkWithDrift. El parámetro estacional usado en el proyecto para SeasonalNaive fue 208 semanas. Estos modelos no buscan ser la solución final sino una línea base interpretable contra la cual comparar modelos futuros. El período de prueba incluye años recientes con brotes y cambios de nivel, por lo cual la evaluación es exigente, un buen baseline debe capturar el nivel general sin sobreajustarse a un episodio específico.

In [24]:
sf, preds = entrenar_y_predecir_baselines(
    train=train,
    test=test,
    frecuencia=FRECUENCIA,
    estacionalidad=ESTACIONALIDAD_CUATRIENAL
)

test_preds = unir_predicciones_con_test(test, preds)

Pronósticos generados:
  Modelos: ['Naive', 'SeasonalNaive', 'WindowAverage', 'RWD']
  Horizonte: 221 semanas


,unique_id,ds,Naive,SeasonalNaive,WindowAverage,RWD
0,dengue_cali,2020-01-06,162.0,216.0,141.666667,162.287356
1,dengue_cali,2020-01-13,162.0,281.0,141.666667,162.574713
2,dengue_cali,2020-01-20,162.0,272.0,141.666667,162.862069
3,dengue_cali,2020-01-27,162.0,310.0,141.666667,163.149425
4,dengue_cali,2020-02-03,162.0,365.0,141.666667,163.436782


In [25]:
graficar_pronosticos_baseline(
    serie=df,
    test=test,
    test_preds=test_preds,
    fecha_corte=FECHA_CORTE,
    modelos_col=MODELOS_BASELINE
)

### ¿Qué observar en los pronósticos?

La comparación visual ayuda a identificar si los modelos baseline capturan el nivel general de la serie, si reaccionan demasiado lento ante brotes o si sobreestiman/subestiman los casos durante el período de prueba.

- `Naive` es útil como referencia mínima: si un modelo complejo no supera repetir el último valor observado, no está aportando información real.
- `SeasonalNaive` evalúa si la observación de hace 208 semanas es una aproximación razonable. En dengue puede funcionar cuando hay estacionalidad multianual, pero puede fallar si un brote cambia de intensidad o de calendario.
- `WindowAverage` suaviza fluctuaciones recientes. Suele ser estable, aunque tiende a retrasarse frente a aumentos abruptos.
- `RandomWalkWithDrift` incorpora una tendencia lineal simple. Puede ser engañoso en epidemias, porque los brotes rara vez evolucionan como una pendiente constante.

La lectura visual debe concentrarse en los períodos de aceleración y desaceleración. Un baseline puede tener error promedio aceptable y, aun así, ser poco útil para vigilancia si llega tarde a los picos o si subestima sistemáticamente el ascenso de casos.


---
## 📊 PARTE 10: Métricas de Evaluación

Se calculan métricas de error para comparar los modelos baseline.

| Métrica | Interpretación |
|---|---|
| MAE | Error absoluto promedio en casos |
| RMSE | Penaliza más los errores grandes |
| sMAPE | Error porcentual simétrico |
| MASE | Error relativo frente a un modelo naive |

In [26]:
evaluacion = calcular_metricas_baseline(
    train=train,
    test_preds=test_preds,
    modelos_col=MODELOS_BASELINE
)

mejor_modelo = seleccionar_mejor_modelo(
    evaluacion=evaluacion,
    modelos_col=MODELOS_BASELINE,
    metrica="mae"
)

print(f"\n✅ Mejor modelo seleccionado por MAE: {mejor_modelo}")


MAE del Naive sobre train (denominador MASE): 12.59
  RESULTADOS POR MÉTRICA — MODELOS BASELINE


,unique_id,metric,Naive,SeasonalNaive,WindowAverage,RWD
0,dengue_cali,mae,111.987330,81.828959,99.925490,130.194695
1,dengue_cali,rmse,123.140951,121.823961,115.617042,140.111914
2,dengue_cali,smape,0.455584,0.590086,0.428548,0.482878
3,dengue_cali,mase,8.894916,6.499500,7.936869,10.341088



📌 Mejor modelo por métrica:
      MAE: SeasonalNaive  (81.8290)
     RMSE: WindowAverage  (115.6170)
    SMAPE: WindowAverage  (0.4285)
     MASE: SeasonalNaive  (6.4995)

✅ Mejor modelo seleccionado por MAE: SeasonalNaive


In [27]:
graficar_metricas(evaluacion, MODELOS_BASELINE)

### Conclusión de métricas

El mejor baseline debe seleccionarse con base en la métrica principal del análisis. En este notebook se usa MAE como criterio principal porque es fácil de interpretar en unidades originales: número de casos de dengue.

Con el corte temporal usado en el notebook, los baselines tienden a mostrar desempeños distintos según la métrica. En una evaluación de referencia sobre los datos locales, `SeasonalNaive` obtiene el menor MAE, mientras que `WindowAverage` puede ser competitivo en RMSE y sMAPE por su comportamiento más suavizado. Esta diferencia es importante: **no existe un ganador absoluto independiente del objetivo**.

La conclusión metodológica es que los baselines cumplen su función: establecen una referencia cuantitativa mínima y muestran que la serie contiene estructura explotable. Sin embargo, sus errores durante cambios bruscos de nivel dejan entrever la necesidad de buscar modelos más completos.


---
## 🔬 PARTE 11: Cierre del Ciclo — Residuos del Mejor Baseline

Después de seleccionar el mejor baseline, se analizan sus residuos. Si los residuos todavía tienen autocorrelación, significa que el modelo dejó patrones temporales sin explicar y que hay margen para modelos más avanzados.

### Lectura esperada de los residuos

El análisis de residuos conecta evaluación predictiva con diagnóstico estadístico. Un modelo puede tener el menor MAE entre los baselines y, aun así, dejar residuos estructurados. En ese caso, el modelo es el mejor dentro del conjunto probado, pero no necesariamente suficiente para representar la dinámica de la enfermedad.

Para un baseline de dengue, residuos positivos persistentes indican subestimación sostenida de casos, típicamente durante fases de brote. Residuos negativos persistentes indican sobreestimación, frecuente después de que un brote empieza a descender. Si el test de Ljung-Box sobre residuos rechaza ruido blanco, la conclusión es que todavía queda memoria temporal no capturada y se justifica avanzar hacia SARIMA, modelos con rezagos, ensambles o enfoques con covariables climáticas.

El gráfico resume tres dimensiones: evolución temporal del error, distribución de residuos y autocorrelación remanente. Si aparecen patrones persistentes, el baseline todavía deja estructura sin explicar.


In [28]:
residuos = diagnosticar_residuos(
    test_preds=test_preds,
    mejor_modelo=mejor_modelo,
    lags_residuos=[1, 6, 12]
)

Residuos del modelo SeasonalNaive
Media: 46.32  |  Desviación estándar: 112.68
Esperado en ruido blanco: media ≈ 0 y desviación estándar relativamente constante

LJUNG-BOX SOBRE RESIDUOS DEL SeasonalNaive:
──────────────────────────────────────────────────────────
  Lag  1: p=0.0000  →  Quedan patrones ⚠️  → margen de mejora
  Lag  6: p=0.0000  →  Quedan patrones ⚠️  → margen de mejora
  Lag 12: p=0.0000  →  Quedan patrones ⚠️  → margen de mejora


---
## 🔁 PARTE 12: Transformaciones de la Serie

Las transformaciones no reemplazan a los baselines iniciales, pero ayudan a diagnosticar y preparar la serie para modelos más exigentes.

- `log1p(y)`: reduce asimetría y estabiliza varianza.
- `diff(y)`: remueve cambios de nivel o tendencia.
- `diff(log1p(y))`: aproxima cambios relativos entre semanas.

In [29]:
df_transformado = crear_transformaciones(df)

graficar_transformaciones(df_transformado)

In [30]:
print("Pruebas de estacionariedad sobre transformaciones:")

r_log = test_estacionaridad(
    df_transformado["y_log"].dropna(),
    "Transformación log1p(y)"
)

r_diff = test_estacionaridad(
    df_transformado["y_diff"].dropna(),
    "Diferencia de la serie diff(y)"
)

r_log_diff = test_estacionaridad(
    df_transformado["y_log_diff"].dropna(),
    "Diferencia logarítmica diff(log1p(y))"
)

Pruebas de estacionariedad sobre transformaciones:

══════════════════════════════════════════════════════════
  Transformación log1p(y)
══════════════════════════════════════════════════════════
  ADF:  stat= -3.2163  p=0.0191 → ES estacionaria ✅
  KPSS: stat=  0.1951  p=0.1000 → ES estacionaria ✅
  ──────────────────────────────────────────────────────
  CONCLUSIÓN: ✅  ESTACIONARIA

══════════════════════════════════════════════════════════
  Diferencia de la serie diff(y)
══════════════════════════════════════════════════════════
  ADF:  stat= -7.8762  p=0.0000 → ES estacionaria ✅
  KPSS: stat=  0.0235  p=0.1000 → ES estacionaria ✅
  ──────────────────────────────────────────────────────
  CONCLUSIÓN: ✅  ESTACIONARIA

══════════════════════════════════════════════════════════
  Diferencia logarítmica diff(log1p(y))
══════════════════════════════════════════════════════════
  ADF:  stat= -7.1690  p=0.0000 → ES estacionaria ✅
  KPSS: stat=  0.1076  p=0.1000 → ES estacionaria ✅
  ─────

### Interpretación de Transformaciones

La transformación `log1p(y)` es especialmente razonable en conteos epidemiológicos porque reduce el peso visual y estadístico de los brotes extremos sin descartar la información. La diferencia simple captura cambios absolutos entre semanas, la diferencia logarítmica aproxima cambios relativos, lo que puede ser más interpretable cuando la serie pasa de niveles bajos a altos.

---
## 🔄 PARTE 13: Time Series Cross-Validation

El split simple de train/test depende de un único corte temporal. La validación cruzada temporal evalúa los modelos en varios cortes, respetando siempre el orden del tiempo.

### Por qué la validación temporal cambia la lectura

En problemas epidemiológicos, el error promedio de un solo período puede ser insuficiente. Una validación cruzada temporal permite evaluar modelos bajo varios estados de la serie: ascenso, descenso, estabilidad relativa y brote. Esto ayuda a distinguir modelos que solo funcionan en un escenario de modelos que conservan desempeño aceptable en contextos diferentes.


In [31]:
cv_results = ejecutar_validacion_cruzada(
    sf=sf,
    serie=df,
    h=52,
    n_windows=3,
    step_size=52
)

Resultados CV: 156 filas × 8 columnas
Folds (cutoffs): [Timestamp('2021-03-29 00:00:00'), Timestamp('2022-03-28 00:00:00'), Timestamp('2023-03-27 00:00:00')]


,unique_id,ds,cutoff,y,Naive,SeasonalNaive,WindowAverage,RWD
0,dengue_cali,2021-04-05,2021-03-29,60.0,69.0,12.0,78.0,69.097104
1,dengue_cali,2021-04-12,2021-03-29,77.0,69.0,18.0,78.0,69.194208
2,dengue_cali,2021-04-19,2021-03-29,51.0,69.0,9.0,78.0,69.291312
3,dengue_cali,2021-04-26,2021-03-29,46.0,69.0,13.0,78.0,69.388416
4,dengue_cali,2021-05-03,2021-03-29,39.0,69.0,11.0,78.0,69.485520
5,dengue_cali,2021-05-10,2021-03-29,53.0,69.0,7.0,78.0,69.582624


In [32]:
cv_eval = evaluar_validacion_cruzada(cv_results, MODELOS_BASELINE)

graficar_validacion_cruzada(
    serie=df,
    cv_results=cv_results,
    modelo=mejor_modelo if mejor_modelo in cv_results.columns else "SeasonalNaive"
)

Métricas promedio en Cross-Validation:
  unique_id     cutoff metric      Naive  SeasonalNaive  WindowAverage        RWD
dengue_cali 2021-03-29    mae  15.769231      48.057692      22.884615  17.873804
dengue_cali 2022-03-28    mae  13.961538      29.461538      11.000000  14.853738
dengue_cali 2023-03-27    mae 154.669231     217.234615     159.938462 153.836992
dengue_cali 2021-03-29   rmse  18.828170      49.859225      25.780881  21.153887
dengue_cali 2022-03-28   rmse  15.675802      31.017365      12.656035  16.732381
dengue_cali 2023-03-27   rmse 199.605800     242.861789     206.144738 198.168276
dengue_cali 2021-03-29  smape   0.135143       0.747551       0.181336   0.149228
dengue_cali 2022-03-28  smape   0.176543       0.699522       0.145776   0.185040
dengue_cali 2023-03-27  smape   0.523267       0.708155       0.554802   0.517781


### 🔍 Interpretación — Cross-Validation

- Cada fold usa un período diferente de entrenamiento y prueba.
- Las métricas promedio son más robustas que las de un único split.
- Si un modelo gana en el split simple pero no en validación cruzada, conviene revisar su estabilidad.

La validación rolling-origin es especialmente importante en dengue porque los brotes no se distribuyen de forma uniforme en el tiempo. Un único holdout puede favorecer accidentalmente al modelo que mejor se ajusta a un brote específico o a un período de baja transmisión. Evaluar varios cortes temporales permite observar si el desempeño se sostiene cuando cambia el contexto epidemiológico.

---
## ✅ Conclusiones

1. **La serie es apta para modelado temporal**, porque puede organizarse como una secuencia semanal regular en formato Nixtla. La calidad temporal básica es suficiente para entrenar y evaluar modelos, siempre que la imputación se haga sin fuga de información.

2. **El dengue en Cali muestra dinámica epidémica y no ruido aleatorio.** La alta autocorrelación de corto plazo confirma persistencia: los niveles de una semana dependen fuertemente de las semanas anteriores. Esto justifica el uso de modelos de series temporales.

3. **Los valores extremos son epidemiológicamente relevantes.** Los picos de 2010, 2020 y 2023 no deben tratarse automáticamente como errores. Representan fases críticas del fenómeno y son precisamente los eventos que un modelo predictivo debería aprender a anticipar.

4. **Los baselines son necesarios, pero insuficientes.** Modelos como `Naive`, `SeasonalNaive`, `WindowAverage` y `RandomWalkWithDrift` entregan una referencia clara de desempeño. Sin embargo, su dificultad para capturar cambios bruscos de régimen muestra que el problema requiere modelos con mayor capacidad dinámica.

5. **La evaluación debe priorizar estabilidad temporal.** Un modelo puede ganar en un holdout específico y perder consistencia en validación rolling-origin. En la práctica, para una aplicación de un modelo en salud pública, la robustez ante distintos períodos es tan importante como el error promedio.

6. **Siguiente paso recomendado:** comparar estos baselines contra modelos como SARIMA/SARIMAX o modelos de machine learning con rezagos y, además, tener en cuenta variables climáticas.
